Input (flattened image)

 → Hidden Layer 1 (ReLU)
 
 → Hidden Layer 2 (ReLU)
 
 → Output Layer (Softmax)
 
 → Cross-Entropy Loss
 
 → Backpropagation


In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
print("Imported successfully")

Imported successfully


In [17]:
def relu(Z):
    return np.maximum(0, Z)


In [18]:
def relu_derivative(Z):
    return (Z > 0).astype(float)


In [19]:
def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)


In [20]:
def cross_entropy_loss(y_true, y_pred):
    m = y_true.shape[0]
    loss = -np.sum(y_true * np.log(y_pred + 1e-8)) / m
    return loss


In [21]:
class NeuralNetwork:
    def __init__(self, input_size, hidden1, hidden2, output_size, lr=0.01):
        self.lr = lr

        # Weights
        self.W1 = np.random.randn(input_size, hidden1) * 0.01
        self.b1 = np.zeros((1, hidden1))

        self.W2 = np.random.randn(hidden1, hidden2) * 0.01
        self.b2 = np.zeros((1, hidden2))

        self.W3 = np.random.randn(hidden2, output_size) * 0.01
        self.b3 = np.zeros((1, output_size))

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = relu(self.Z1)

        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = relu(self.Z2)

        self.Z3 = np.dot(self.A2, self.W3) + self.b3
        self.A3 = softmax(self.Z3)

        return self.A3

    def backward(self, X, y_true):
        m = X.shape[0]

        dZ3 = self.A3 - y_true
        dW3 = np.dot(self.A2.T, dZ3) / m
        db3 = np.sum(dZ3, axis=0, keepdims=True) / m

        dA2 = np.dot(dZ3, self.W3.T)
        dZ2 = dA2 * relu_derivative(self.Z2)
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * relu_derivative(self.Z1)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # Update weights
        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, epochs=300):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = cross_entropy_loss(y, y_pred)
            self.backward(X, y)
            if epoch % 50 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")


    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)





## STEP 1. Dataset Handling & Preprocessing

### 1.1 Unzip the Files

In [22]:
import zipfile
import os

# unzip training data
with zipfile.ZipFile("Train.zip", 'r') as zip_ref:
    zip_ref.extractall("train")

# unzip test data
with zipfile.ZipFile("Test.zip", 'r') as zip_ref:
    zip_ref.extractall("test")


This creates two folders:
train/ and test/

In [23]:
print(os.listdir("train"))


['Train']


In [24]:
print(os.listdir("train/Train"))


['Jade', 'James', 'Jane', 'Joel', 'Jovi']


### 1.2  Label Encoding

In [25]:
import os

train_dir = "train/Train"

class_names = sorted(os.listdir(train_dir))

label_map = {class_name: idx for idx, class_name in enumerate(class_names)}
print(label_map)



{'Jade': 0, 'James': 1, 'Jane': 2, 'Joel': 3, 'Jovi': 4}


## 1.3 Image Loading & Preprocessing

In [26]:
from PIL import Image
import numpy as np

def load_and_preprocess_image(img_path, img_size=(64, 64)):
    img = Image.open(img_path).convert("L")  # grayscale
    img = img.resize(img_size)
    img_array = np.array(img)
    img_array = img_array / 255.0            # normalize
    return img_array.flatten()               # flatten


Why grayscale?

Teeth shape > color

Less parameters

Faster training

In [27]:
X_train = []
y_train = []

for class_name in class_names:
    class_folder = os.path.join(train_dir, class_name)
    
    for img_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_name)
        
        X_train.append(load_and_preprocess_image(img_path))
        y_train.append(label_map[class_name])


In [28]:
X_train = np.array(X_train)
y_train = np.array(y_train)


In [29]:
def one_hot_encode(y, num_classes):
    one_hot = np.zeros((y.size, num_classes))
    one_hot[np.arange(y.size), y] = 1
    return one_hot

y_train_oh = one_hot_encode(y_train, num_classes=len(class_names))


In [30]:
X_train.shape

(4000, 4096)

In [31]:
y_train_oh.shape


(4000, 5)

## Training the Neural Network

### Initialize the model

In [32]:
input_size = X_train.shape[1]      # should be 4096 if 64x64
output_size = y_train_oh.shape[1]  # should be 5

print(input_size, output_size)


4096 5


In [34]:
nn = NeuralNetwork(
    input_size=input_size,
    hidden1=128,
    hidden2=64,
    output_size=output_size,
    lr=0.03
)


### Train the model

In [35]:
nn.train(X_train, y_train_oh, epochs=800)


Epoch 0, Loss: 1.6094
Epoch 50, Loss: 1.6087
Epoch 100, Loss: 1.6072
Epoch 150, Loss: 1.6024
Epoch 200, Loss: 1.5774
Epoch 250, Loss: 1.4569
Epoch 300, Loss: 1.2957
Epoch 350, Loss: 1.2011
Epoch 400, Loss: 1.0399
Epoch 450, Loss: 0.8526
Epoch 500, Loss: 0.6938
Epoch 550, Loss: 0.5925
Epoch 600, Loss: 0.5414
Epoch 650, Loss: 0.5273
Epoch 700, Loss: 0.4488
Epoch 750, Loss: 0.4106


### Evaluate Training Accuracy

In [36]:
train_preds = nn.predict(X_train)
true_labels = np.argmax(y_train_oh, axis=1)

accuracy = np.mean(train_preds == true_labels)
print(f"Training Accuracy: {accuracy * 100:.2f}%")


Training Accuracy: 88.92%


In [37]:
print(os.listdir("test"))


['Test']


In [38]:
##test

In [39]:
test_dir = "test/Test"   

X_test = []
y_test = []

for class_name in class_names:
    class_folder = os.path.join(test_dir, class_name)
    
    for img_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_name)
        X_test.append(load_and_preprocess_image(img_path))
        y_test.append(label_map[class_name])

X_test = np.array(X_test)
y_test = np.array(y_test)


In [40]:
test_preds = nn.predict(X_test)

In [41]:
test_accuracy = np.mean(test_preds == y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


Test Accuracy: 87.40%
